# Call the C-MAPSS real-time endpoint

Use the project's Python 3.12 `.venv` kernel. The endpoint uses **Microsoft Entra authentication**, not an endpoint key. No subscription ID, resource group, workspace config, or storage upload is needed for this call. The request sends the local example records as JSON and receives predictions directly.

Authentication uses your Azure CLI login. If needed, run `az cloud set --name AzureUSGovernment` and then `az login` in a terminal. Your signed-in identity must have permission `Microsoft.MachineLearningServices/workspaces/onlineEndpoints/score/action` on the endpoint (or an enclosing scope). A 403 response can indicate missing scoring permission.

The endpoint must have a healthy deployment receiving traffic, with `deployment/score.py` and an inference environment matching the registered model. Real-time compute incurs charges while deployed, even when idle.

The URL below comes from your screenshot. The submit cell obtains a short-lived Government ML token automatically. It never prompts for or prints a key or token.

## 1. Preview the request

These two records come from the Python 3.10/scikit-learn 1.5 model's saved example input. No local model is loaded.

Source: A. Saxena and K. Goebel (2008), NASA C-MAPSS. Simulation predictions are not certified maintenance advice.

In [1]:
import json
from pathlib import Path

import requests
from azure.identity import AzureCliCredential

ENDPOINT_URL = "https://aml-demo-dtyps.usgovvirginia.inference.ml.azure.us/score"
DEPLOYMENT_NAME = "cmapss-fd001-local-regressor-2"
ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "pyproject.toml").is_file())
EXAMPLE_PATH = ROOT / "models" / "cmapss-fd001-sklearn15" / "example_input.json"
request_body = json.loads(EXAMPLE_PATH.read_text(encoding="utf-8"))
request_headers = {
    "Content-Type": "application/json",
    "azureml-model-deployment": DEPLOYMENT_NAME,
}
credential = AzureCliCredential()

print("POST", ENDPOINT_URL)
print(json.dumps({**request_headers, "Authorization": "Bearer <redacted>"}, indent=2))
print(json.dumps(request_body, indent=2))

POST https://aml-demo-dtyps.usgovvirginia.inference.ml.azure.us/score
{
  "Content-Type": "application/json",
  "azureml-model-deployment": "cmapss-fd001-local-regressor-2",
  "Authorization": "Bearer <redacted>"
}
[
  {
    "cycle": 31,
    "setting_1": -0.0006,
    "setting_2": 0.0004,
    "setting_3": 100.0,
    "t2_fan_inlet_total_temperature_rankine": 518.67,
    "t24_lpc_outlet_total_temperature_rankine": 642.58,
    "t30_hpc_outlet_total_temperature_rankine": 1581.22,
    "t50_lpt_outlet_total_temperature_rankine": 1398.91,
    "p2_fan_inlet_pressure_psia": 14.62,
    "p15_bypass_duct_total_pressure_psia": 21.61,
    "p30_hpc_outlet_total_pressure_psia": 554.42,
    "nf_physical_fan_speed_rpm": 2388.08,
    "nc_physical_core_speed_rpm": 9056.4,
    "epr_engine_pressure_ratio": 1.3,
    "ps30_hpc_outlet_static_pressure_psia": 47.23,
    "phi_fuel_flow_to_hpc_static_pressure_ratio": 521.79,
    "nrf_corrected_fan_speed_rpm": 2388.06,
    "nrc_corrected_core_speed_rpm": 8130.11,
  

## 2. Send the API request

Run this cell to send the JSON directly to `/score`. It obtains a Microsoft Entra token from your Azure CLI login for `https://ml.azure.us/.default` and sends it as a bearer token. No endpoint key is used. The request is not automatically retried. The next cell shows the response body, including server errors.

In [2]:
response = None
token = credential.get_token("https://ml.azure.us/.default")
try:
    response = requests.post(
        ENDPOINT_URL,
        headers={**request_headers, "Authorization": f"Bearer {token.token}"},
        json=request_body,
        timeout=120,
        allow_redirects=False,
    )
except (requests.ConnectionError, requests.Timeout):
    raise RuntimeError("No response received. Check endpoint connectivity and deployment health.") from None
finally:
    del token
    if response is not None:
        response.request.headers["Authorization"] = "Bearer <redacted>"

## 3. Display the API response

A successful call returns `predictions` and `unit: cycles`. For the retrained model, the two saved examples should produce approximately **183.763224** and **144.302739** cycles. A different deployed model version may return different values.

If scoring fails, inspect the printed response and the deployment logs. Confirm a deployment exists, receives traffic, uses the matching inference environment, and has the updated scoring script.

In [3]:
if response is None:
    raise RuntimeError("Run the request cell first and check for connection errors.")
print("HTTP", response.status_code)
try:
    response_body = response.json()
    print(json.dumps(response_body, indent=2))
except ValueError:
    response_body = None
    print(response.text)
response.raise_for_status()
if not 200 <= response.status_code < 300:
    raise RuntimeError("Unexpected non-success response; inspect the body above.")
if not isinstance(response_body, dict) or not isinstance(response_body.get("predictions"), list):
    raise RuntimeError("Expected a predictions list; check the deployed scoring script.")
if len(response_body["predictions"]) != len(request_body):
    raise RuntimeError("Prediction count does not match the input record count.")

HTTP 200
{
  "predictions": [
    183.7632235282114,
    144.3027393871352
  ],
  "unit": "cycles"
}
